# Model Evaluation — Toxicity Detection

Evaluates all five models on the **dev** split (labelled) and produces a side-by-side comparison.
Test split has no labels so only predictions are saved for it.

| Key | Model |
|-----|-------|
| `baseline` | TF-IDF + Logistic Regression (original train) |
| `baseline_aug` | TF-IDF + Logistic Regression (augmented train) |
| `xlm_roberta` | XLM-RoBERTa fine-tuned on original train |
| `xlm_roberta_aug` | XLM-RoBERTa fine-tuned on augmented train |
| `bert_multilingual` | bert-base-multilingual-cased fine-tuned on original train |

## 1) Imports & config

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import spacy

from pathlib import Path
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR            = Path("/path/to/project/data")
TRAIN_PATH          = DATA_DIR / "train_baseline.csv"
AUG_TRAIN_PATH      = DATA_DIR / "train_aug_baseline.csv"
DEV_PATH            = DATA_DIR / "dev_baseline.csv"
TEST_PATH           = DATA_DIR / "test_baseline.csv"

XLM_MODEL_PATH      = DATA_DIR / "xlm_final"
XLM_AUG_MODEL_PATH  = DATA_DIR / "xlm_final_augmented"
BERT_MODEL_PATH     = DATA_DIR / "bert_final"

LANGUAGES = ["en", "de", "fi"]

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2) Load preprocessed data

In [2]:
train_df         = pd.read_csv(TRAIN_PATH)
aug_train_df     = pd.read_csv(AUG_TRAIN_PATH)
dev_df           = pd.read_csv(DEV_PATH)
test_df          = pd.read_csv(TEST_PATH)

# Fill any NaN preprocessed text
for df in [train_df, aug_train_df, dev_df, test_df]:
    df["text_baseline"] = df["text_baseline"].fillna("")

y_train     = train_df["label"].astype(int)
y_aug_train = aug_train_df["label"].astype(int)
y_dev       = dev_df["label"].astype(int)

print(f"train: {len(train_df):,}  aug_train: {len(aug_train_df):,}  dev: {len(dev_df):,}  test: {len(test_df):,}")
print("Dev language distribution:")
print(dev_df["language"].value_counts())

train: 99,000  aug_train: 113,850  dev: 13,200  test: 12,791
Dev language distribution:
language
en    11000
de     2000
fi      200
Name: count, dtype: int64


## 3) Shared metric utilities

In [3]:
def _pos_proba(y_proba):
    """Return 1-D positive-class probability array."""
    arr = np.asarray(y_proba)
    if arr.ndim == 2:
        return arr[:, 1] if arr.shape[1] >= 2 else arr[:, 0]
    return arr


def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    try:
        auroc = roc_auc_score(y_true, _pos_proba(y_proba))
    except Exception:
        auroc = np.nan
    return {
        "f1":        f1_score(y_true, y_pred, average="weighted"),
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "auroc":     auroc,
    }


def evaluate_with_languages(df, preds, probs):
    """Return overall + per-language metric dict."""
    preds = np.asarray(preds)
    probs = np.asarray(probs)
    out = {"overall": compute_metrics(df["label"], preds, probs), "by_language": {}}
    for lang in LANGUAGES:
        mask = (df["language"] == lang).to_numpy()
        if mask.sum() == 0:
            out["by_language"][lang] = {k: np.nan for k in ["f1","accuracy","precision","recall","auroc"]}
        else:
            out["by_language"][lang] = compute_metrics(df["label"].to_numpy()[mask], preds[mask], probs[mask])
    return out


def metrics_to_df(metrics_by_model):
    """Flatten {model_name: metrics_dict} → DataFrame."""
    rows = []
    for model_name, m in metrics_by_model.items():
        rows.append({"model": model_name, "subset": "overall", **m["overall"]})
        for lang, lm in m["by_language"].items():
            rows.append({"model": model_name, "subset": lang, **lm})
    return pd.DataFrame(rows)


print("Metric utilities ready.")

Metric utilities ready.


## 4) Baseline models (TF-IDF + Logistic Regression)

In [4]:
results = {}

for key, X_train_text, y_train_labels in [
    ("baseline",     train_df["text_baseline"],     y_train),
    ("baseline_aug", aug_train_df["text_baseline"],  y_aug_train),
]:
    vec = TfidfVectorizer(max_features=5000)
    X_tr = vec.fit_transform(X_train_text)
    X_dv = vec.transform(dev_df["text_baseline"])
    X_te = vec.transform(test_df["text_baseline"])

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_tr, y_train_labels)

    dev_preds = clf.predict(X_dv)
    dev_probs = clf.predict_proba(X_dv)          # shape (N, 2)
    test_preds = clf.predict(X_te)
    test_probs = clf.predict_proba(X_te)

    results[key] = {
        "dev_preds":  dev_preds,
        "dev_probs":  dev_probs,
        "test_preds": test_preds,
        "test_probs": test_probs,
        "metrics":    evaluate_with_languages(dev_df, dev_preds, dev_probs),
    }
    acc = accuracy_score(y_dev, dev_preds)
    print(f"{key} dev accuracy: {acc:.4f}")

print("Baselines done.")

baseline dev accuracy: 0.8781
baseline_aug dev accuracy: 0.8776
Baselines done.


## 5) Transformer models (load from disk, inference only)

In [5]:
def build_hf_dataset(df, tokenizer, max_length=128, include_labels=False):
    cols = ["text"] + (["label"] if include_labels else [])
    ds = Dataset.from_pandas(df[cols].copy())
    if include_labels:
        ds = ds.rename_column("label", "labels")

    def _tok(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=max_length)

    ds = ds.map(_tok, batched=True)
    keep = ["input_ids", "attention_mask"] + (["labels"] if include_labels else [])
    if "token_type_ids" in ds.column_names:
        keep.append("token_type_ids")
    ds.set_format(type="torch", columns=keep)
    return ds


def evaluate_transformer(model_path, result_key):
    print(f"\nLoading {result_key} from {model_path} …")
    tok   = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    dev_ds  = build_hf_dataset(dev_df,  tok, include_labels=True)
    test_ds = build_hf_dataset(test_df, tok, include_labels=False)

    trainer = Trainer(model=model)
    dev_out  = trainer.predict(dev_ds)
    test_out = trainer.predict(test_ds)

    def _decode(outputs):
        logits = outputs.predictions
        probs  = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()
        preds  = np.argmax(probs, axis=1)
        return preds, probs

    dev_preds,  dev_probs  = _decode(dev_out)
    test_preds, test_probs = _decode(test_out)

    results[result_key] = {
        "dev_preds":  dev_preds,
        "dev_probs":  dev_probs,
        "test_preds": test_preds,
        "test_probs": test_probs,
        "metrics":    evaluate_with_languages(dev_df, dev_preds, dev_probs),
    }

    acc = accuracy_score(y_dev, dev_preds)
    print(f"  dev accuracy: {acc:.4f}")


evaluate_transformer(XLM_MODEL_PATH,     "xlm_roberta")
evaluate_transformer(XLM_AUG_MODEL_PATH, "xlm_roberta_aug")
evaluate_transformer(BERT_MODEL_PATH,    "bert_multilingual")

print("\nAll transformer models evaluated.")


Loading xlm_roberta from /path/to/project/data/xlm_final …


Map: 100%|██████████| 12791/12791 [00:00<00:00, 33598.56 examples/s]
/path/to/project/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  dev accuracy: 0.9120

Loading xlm_roberta_aug from /path/to/project/data/xlm_final_augmented …


Map: 100%|██████████| 12791/12791 [00:00<00:00, 32244.80 examples/s]
/path/to/project/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  dev accuracy: 0.9126

Loading bert_multilingual from /path/to/project/data/bert_final …


Map: 100%|██████████| 12791/12791 [00:00<00:00, 24410.81 examples/s]
/path/to/project/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  dev accuracy: 0.9072

All transformer models evaluated.


## 6) Results comparison

In [6]:
MODEL_ORDER = ["baseline", "baseline_aug", "xlm_roberta", "xlm_roberta_aug", "bert_multilingual"]

all_metrics = {k: results[k]["metrics"] for k in MODEL_ORDER}
comparison_df = metrics_to_df(all_metrics)

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

overall_df = comparison_df[comparison_df["subset"] == "overall"].reset_index(drop=True)
lang_df    = comparison_df[comparison_df["subset"] != "overall"].reset_index(drop=True)

print("=" * 80)
print("OVERALL METRICS (dev split)")
print("=" * 80)
print(overall_df.to_string(index=False))

print()
print("=" * 80)
print("PER-LANGUAGE METRICS (dev split)")
print("=" * 80)
print(lang_df.to_string(index=False))

OVERALL METRICS (dev split)
            model  subset     f1  accuracy  precision  recall  auroc
         baseline overall 0.8745    0.8781     0.8801  0.8781 0.9190
     baseline_aug overall 0.8749    0.8776     0.8777  0.8776 0.9367
      xlm_roberta overall 0.9122    0.9120     0.9125  0.9120 0.9674
  xlm_roberta_aug overall 0.9133    0.9126     0.9154  0.9126 0.9693
bert_multilingual overall 0.9069    0.9072     0.9067  0.9072 0.9648

PER-LANGUAGE METRICS (dev split)
            model subset     f1  accuracy  precision  recall  auroc
         baseline     en 0.9126    0.9135     0.9135  0.9135 0.9670
         baseline     de 0.6747    0.7455     0.6837  0.7455 0.5849
         baseline     fi 0.1188    0.2550     0.5622  0.2550 0.5463
     baseline_aug     en 0.9101    0.9111     0.9110  0.9111 0.9659
     baseline_aug     de 0.7101    0.7415     0.7035  0.7415 0.6541
     baseline_aug     fi 0.3765    0.3950     0.7237  0.3950 0.5949
      xlm_roberta     en 0.9398    0.9395     0.

## 7) Save test predictions for all models

In [ ]:
out_dir = DATA_DIR / "predictions"
out_dir.mkdir(exist_ok=True)

for key in MODEL_ORDER:
    df_out = pd.DataFrame({
        "id":        test_df["id"],
        "predicted": results[key]["test_preds"],
    })
    path = out_dir / f"test_predictions_{key}.tsv"
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Saved: {path}")

# Also save the comparison tables
overall_df.to_csv(out_dir / "overall_comparison.csv", index=False)
lang_df.to_csv(out_dir / "per_language_comparison.csv", index=False)
print("Comparison CSVs saved.")